# Практика § 4.2: рамки, маски и границы

Выполняйте ячейки сверху вниз. Сначала считайте площади и пересечения вручную.

## IoU рамок

Предскажите результат для одинаковых, непересекающихся и частично пересекающихся рамок.

In [ ]:
def area(box):
    x1, y1, x2, y2 = box
    return max(0, x2-x1) * max(0, y2-y1)

def iou(a, b):
    overlap = (max(a[0],b[0]), max(a[1],b[1]),
               min(a[2],b[2]), min(a[3],b[3]))
    intersection = area(overlap)
    union = area(a) + area(b) - intersection
    return intersection / union if union else 0.0

truth = (0,0,4,4)
for predicted in [(0,0,4,4), (2,0,6,4), (5,0,7,2)]:
    print(predicted, round(iou(truth, predicted), 3))

## Когда NMS может ошибиться

Две рамки одного класса сильно пересекаются. Измените порог подавления: какая рамка исчезает? Затем представьте две настоящие чашки рядом и объясните риск слишком низкого порога.

In [ ]:
boxes = [(1,1,5,5), (1,1,6,5), (6,1,9,5)]
scores = [0.9, 0.7, 0.8]
for threshold in [0.5, 0.85]:
    kept = []
    for index in sorted(range(len(boxes)), key=lambda i: -scores[i]):
        if all(iou(boxes[index], boxes[old]) <= threshold for old in kept):
            kept.append(index)
    print(threshold, kept)

## Маска пикселей

Рамка охватывает фон; сравните пиксельные множества. Добавьте один ложный пиксель в прогноз и пересчитайте IoU.

In [ ]:
truth_pixels = {1,2,3,4}
predicted_pixels = {2,3,4,5,6}
print(len(truth_pixels & predicted_pixels) / len(truth_pixels | predicted_pixels))

## Самостоятельно

Нарисуйте схему U-Net с формами 64→32→16→32→64. Подпишите раннюю карту, переданную на обратном пути при размере 32. Укажите, где у Mask R-CNN находится отдельная маска экземпляра, а где у U-Net — карта класса каждого пикселя.

## Самопроверка




### 1. Какой правильный ответ нужен для обучения детектора на двух предметах?

A. Одна метка всего кадра

B. Класс и рамка каждого предмета

C. Только размер файла

D. Только цвет фона




### 2. Какова площадь рамки (2,1,6,4)?

A. 7

B. 10

C. 12

D. 24




### 3. Чему равен IoU двух одинаковых непустых рамок?

A. 0

B. 0,5

C. 1

D. Зависит от класса




### 4. Что не проверяет IoU рамок?

A. Совпадение площади

B. Положение области

C. Класс объекта

D. Степень перекрытия




### 5. Зачем применяют NMS?

A. Удалить похожие дубли одного обнаружения

B. Нарисовать маску пикселей

C. Обучить веса с нуля

D. Поменять размер исходного фото




### 6. Какая разметка нужна для обучения маске экземпляра?

A. Только класс кадра

B. Только рамка

C. Пиксели каждого отдельного объекта

D. Только имя файла




### 7. Какую задачу в основном примере решает U-Net?

A. Генерацию подписи

B. Семантическую сегментацию

C. Поиск ближайшего текста

D. Оценку времени печати




### 8. Чем типичная связь U-Net отличается от остаточной связи ResNet?

A. Всегда удаляет карту

B. Соединяет ранние и поздние карты по каналам вместо сложения поправки

C. Меняет правильные метки

D. Не передаёт данные




### 9. Что означает выход (1,3,64,64) в примере пиксельной классификации?

A. Три снимка 64 классов

B. Один снимок, три оценки классов на каждом из 64×64 пикселей

C. Одна рамка

D. 64 маски экземпляров




### 10. Почему случайная сеть из листинга не показывает качество сегментации?

A. У неё нет правильной разметки и обученных весов

B. Она содержит свёртку 1×1

C. Её вход цветной

D. Она считает логиты




## Подсказки и решения

### Вопрос 1

**Подсказка**


Сколько предметов надо указать?



**Решение**


**B.** Размечают каждый объект отдельно.



### Вопрос 2

**Подсказка**


Умножьте ширину на высоту.



**Решение**


**C.** Ширина 4, высота 3, площадь 12.



### Вопрос 3

**Подсказка**


Сравните пересечение и объединение.



**Решение**


**C.** Они совпадают, поэтому отношение равно 1.



### Вопрос 4

**Подсказка**


IoU использует только координаты.



**Решение**


**C.** Рамка может хорошо совпасть и при неверном классе.



### Вопрос 5

**Подсказка**


Что делать с несколькими сильно перекрывающимися рамками?



**Решение**


**A.** NMS оставляет наиболее оценённые кандидаты и убирает дубли.



### Вопрос 6

**Подсказка**


Можно ли узнать форму из прямоугольника?



**Решение**


**C.** Нужна пиксельная форма каждого экземпляра.



### Вопрос 7

**Подсказка**


Что выдаётся для каждого пикселя?



**Решение**


**B.** U-Net строит карту классов пикселей.



### Вопрос 8

**Подсказка**


Вспомните операции со связанными картами.



**Решение**


**B.** U-Net передаёт ранние карты на путь восстановления и обычно конкатенирует их.



### Вопрос 9

**Подсказка**


Расшифруйте оси пакета, каналов и сторон.



**Решение**


**B.** Для каждого пикселя есть три логита классов.



### Вопрос 10

**Подсказка**


Что именно проверяет этот код?



**Решение**


**A.** Он проверяет формы данных, а не точность не обученной сети.



